# K4 — Calibration on numpyro

This page checks phase K4: the original estival + pymc calibration rebuilt on summer4's
`BayesianModel` — the same eighteen uniform priors, the same ten targets with the same
tolerance rule, and a prior-distributed standard deviation on the mixing-matrix distance.

**Claims to check.**

1. At the golden parameters the port's log-likelihood equals estival's (mean over each
   target's years of the normal log-density), computed independently from the golden outputs.
2. Synthetic recovery: targets simulated at known parameters, then a five-parameter AIES
   calibration (gradient-free, the stand-in for `DEMetropolisZ`). Every true value should sit
   inside its 95% posterior interval.
3. A MAP fit with optax (replacing nevergrad) moves the model onto the real targets.

Claims 2 and 3 read `outputs/synthetic_recovery/` and `outputs/find_map/`. Produce them with
`pixi run synthetic-recovery` (about 40 minutes) and `pixi run find-map` (about 10 minutes).
If they are missing this page runs a short version of each, which is enough to see the
claim but not to sign it off.

In [ ]:
import json
import subprocess
import sys

import jax
import numpyro  # noqa: F401 - import before arviz; see docs/summer4-workarounds.md, S4
import arviz as az
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import yaml
from scipy import stats

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from kiribati_tb.calibration import bayesian_model, calibration_setup
from kiribati_tb.paths import GOLDEN, REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tests"))
from parity import read_golden

setup = calibration_setup()

## Priors and targets

The priors are read from `parameters.xlsx`; the bars show each prior's range relative to its
midpoint. The table lists the targets, their observation years and standard deviations
(20% of the mean over 1.96; 40% for notifications).

In [ ]:
priors = pd.DataFrame([(p.name, p.lo, p.hi) for p in setup.priors], columns=["name", "lo", "hi"])
priors["relative width"] = (priors["hi"] - priors["lo"]) / (0.5 * (priors["hi"] + priors["lo"]))
priors.set_index("name")["relative width"].plot.bar(
    title="Uniform priors: width relative to midpoint", labels={"value": "(hi − lo) / midpoint"}
)

In [ ]:
pd.DataFrame(
    [
        {
            "target": t.key,
            "years": f"{int(t.times.min())}–{int(t.times.max())}" if len(t.times) > 1 else int(t.times[0]),
            "observed (mean)": float(np.mean(t.values)),
            "sd": getattr(t.likelihood.sd, "name", t.likelihood.sd),
        }
        for t in setup.targets
    ]
)

## Likelihood equals estival's at the golden parameters

The port's log-likelihood (log density minus log prior) against estival's rule applied to
the golden derived outputs, per target and in total. The two bars in each pair should match.

In [ ]:
params = yaml.safe_load((GOLDEN / "hetero_baseline" / "params.yaml").read_text())["params"]
mix_sd = 12.5
golden = read_golden("hetero_baseline", "derived")
bm = bayesian_model(setup, solver={"solver": "dopri5", "rtol": 1e-10, "atol": 1e-10})
sites = {p.name: params.get(p.name, mix_sd) for p in bm._sites}
per_target = {}
for t in setup.targets:
    sd = t.likelihood.sd if isinstance(t.likelihood.sd, float) else mix_sd
    per_target[t.key] = float(
        np.mean(stats.norm.logpdf(t.values, loc=golden.loc[t.times, t.key].to_numpy(), scale=sd))
    )
estival_total = sum(per_target.values())

import numpyro.distributions.transforms as T

z = bm.unconstrain(sites)
log_prior = sum(
    float(p.to_numpyro().log_prob(sites[p.name]))
    + float(T.biject_to(p.to_numpyro().support).log_abs_det_jacobian(z[p.name], sites[p.name]))
    for p in bm._sites
)
port_total = float(jax.jit(bm.log_density)(z)) - log_prior
print(f"port {port_total:.6f}  estival rule {estival_total:.6f}")
assert abs(port_total - estival_total) <= 1e-6 * abs(estival_total)
pd.Series(per_target).plot.bar(
    title=f"estival log-likelihood per target (total {estival_total:.2f}; port {port_total:.2f})",
    labels={"index": "target", "value": "mean log-density"},
)

## Synthetic recovery

Posterior draws of the five calibrated parameters, with the true value as a red line. The
truth should fall inside every 95% interval.

In [ ]:
out = REPO_ROOT / "outputs" / "synthetic_recovery"
if not (out / "summary.json").exists():
    subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "synthetic_recovery.py"),
         "--warmup", "300", "--samples", "300"],
        check=True,
    )
summary = json.loads((out / "summary.json").read_text())
idata = az.from_netcdf(out / "idata.nc")
draws = pd.DataFrame(
    {name: np.asarray(idata.posterior[name]).reshape(-1) for name in summary["params"]}
)
scaled = pd.DataFrame(
    {
        name: (draws[name] - rec["prior"][0]) / (rec["prior"][1] - rec["prior"][0])
        for name, rec in summary["params"].items()
    }
)
truth = {
    name: (rec["truth"] - rec["prior"][0]) / (rec["prior"][1] - rec["prior"][0])
    for name, rec in summary["params"].items()
}
fig = px.box(
    scaled.melt(var_name="parameter", value_name="position in prior range"),
    x="parameter", y="position in prior range", points=False,
    title="Synthetic recovery: posterior (box) and truth (red), scaled to each prior's range",
)
fig.add_scatter(x=list(truth), y=list(truth.values()), mode="markers",
                marker=dict(color="red", size=12, symbol="x"), name="truth")
fig

In [ ]:
recovery = pd.DataFrame(summary["params"]).T
print(recovery[["truth", "q025", "median", "q975", "covered"]])
assert recovery["covered"].all()

## MAP fit to the real targets

Observed targets against the model at the MAP. The notification series should be followed
in level and shape; the 2024 cross-sectional targets should sit near their observations.

In [ ]:
map_dir = REPO_ROOT / "outputs" / "find_map" / "base"
if not (map_dir / "map.json").exists():
    subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "find_map.py"), "--steps", "60"], check=True
    )
record = json.loads((map_dir / "map.json").read_text())
print(f"log density: start {record['log_density_start']:.1f} -> MAP {record['log_density_map']:.1f}")
assert record["log_density_map"] > record["log_density_start"]
fit = record["fit"]
notif = pd.DataFrame(
    {"observed": fit["notifications"]["observed"], "model at MAP": fit["notifications"]["modelled"]},
    index=fit["notifications"]["times"],
)
notif.plot(title="Notifications: observed vs model at MAP", labels={"index": "year", "value": "notifications"})

In [ ]:
cross = pd.DataFrame(
    {k: {"observed": v["observed"][0], "model at MAP": v["modelled"][0]}
     for k, v in fit.items() if k != "notifications"}
).T
cross["model / observed"] = cross["model at MAP"] / cross["observed"].where(cross["observed"] != 0)
cross["model / observed"].plot.bar(
    title="Cross-sectional targets at MAP (1 = on target; mixing distance observed at 0)",
    labels={"index": "target", "value": "model / observed"},
)